# Analisi della coordinazione locale in argento FCC

## Obiettivo

Visualizzare e analizzare la struttura cristallina FCC dell'argento, identificando la prima shell di coordinazione per atomi selezionati.

## Prerequisiti

- Python 3.12 o superiore
- OVITO 3.16+ per la visualizzazione
- Package: `ovito`, `numpy`, `ipywidgets`

## Output atteso

1. **Visualizzazione** della struttura cristallina in 4 viste
2. **Analisi numerica** della coordinazione locale
3. **Classificazione** dell'ambiente atomico (bulk, superficie, etc.)
4. **Stima** del parametro reticolare FCC

## 1. Configurazione iniziale

Parametri modificabili per l'analisi.

In [1]:
# =========================================================
# PARAMETRI DELL'ANALISI
# =========================================================

# File da analizzare
FILENAME = "fcc100a864.txt"  # 864 atomi, struttura FCC più grande
# FILENAME = "fcc100a108.txt"  # 108 atomi, struttura più piccola

# Atomo di riferimento (numerazione umana: 1, 2, 3, ...)
REFERENCE_ATOM = 200
# Altri atomi interessanti: 5 (superficie), 75 (bulk), 200 (bulk in struttura grande)

# Intervallo per la prima shell di coordinazione
R_MIN = 2.0  # Å
R_MAX = 3.0  # Å

# Parametri di visualizzazione
BACKGROUND_RADIUS_SCALE = 0.38  # Scala per atomi di contesto
BACKGROUND_TRANSPARENCY = 0.85  # Trasparenza atomi di contesto (0=opaco, 1=trasparente)

## 2. Caricamento delle dipendenze

**IMPORTANTE**: Questa cella aggiunge automaticamente il src/ al path Python.

In [2]:
import sys
from pathlib import Path


# Trova la root del progetto e aggiungi src/ al path
def _setup_path():
    cwd = Path.cwd()
    # Risali finché non trovi pyproject.toml
    root = cwd
    for _ in range(10):  # Massimo 10 livelli
        if (root / "pyproject.toml").exists():
            break
        root = root.parent

    src_path = str(root / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    return root


PROJECT_ROOT = _setup_path()
print(f"Project root: {PROJECT_ROOT}")
print(f"src/ aggiunto al path")

# Ora importa le dipendenze
import numpy as np
import ovito

from computational_course.ovito_visualization import (
    find_project_root,
    load_structure,
    convert_to_xyz,
    compute_neighbor_distances,
    create_ovito_pipeline,
    create_viewport_grid,
    print_analysis_report,
)

print("✓ Import completato.")

Project root: /Users/francesco/Desktop/computational-course
src/ aggiunto al path
✓ Import completato.


## 3. Caricamento della struttura

Leggiamo il file delle coordinate e convertiamo in formato XYZ per OVITO.

In [3]:
# Carichiamo il file
txt_path, coords = load_structure(FILENAME, PROJECT_ROOT)

natoms = len(coords)
print(f"File: {txt_path.name}")
print(f"Atomi letti: {natoms}")
print(f"Shape coordinate: {coords.shape}")

print("\nPrime 5 coordinate:")
for i, (x, y, z) in enumerate(coords[:5]):
    print(f"  {i + 1:3d}: ({x:.4f}, {y:.4f}, {z:.4f}) Å")

File: fcc100a864.txt
Atomi letti: 864
Shape coordinate: (864, 3)

Prime 5 coordinate:
    1: (2.0802, 2.0802, 0.0000) Å
    2: (0.0000, 0.0000, 0.0000) Å
    3: (0.0000, 4.1604, 0.0000) Å
    4: (2.0802, 6.2406, 0.0000) Å
    5: (0.0000, 8.3208, 0.0000) Å


## 4. Conversione in formato OVITO

OVITO richiede file in formato XYZ semplice. Creiamo un file temporaneo.

In [4]:
# Creiamo il file XYZ
xyz_path = convert_to_xyz(coords, txt_path.stem, element="Ag")
print(f"File XYZ creato: {xyz_path}")

File XYZ creato: generated/fcc100a864.xyz


## 5. Analisi geometrica

Calcoliamo distanze, identificiamo i primi vicini e stimiamo parametri geometrici.

In [5]:
# Calcoliamo le informazioni sui vicini
results = compute_neighbor_distances(coords, REFERENCE_ATOM, R_MIN, R_MAX)

# Posizione dell'atomo di riferimento
reference_index = REFERENCE_ATOM - 1
reference_position = coords[reference_index]

# Creiamo la maschera per i vicini (necessaria per OVITO)
neighbor_mask = np.zeros(len(coords), dtype=bool)
neighbor_mask[results["neighbor_indices"]] = True

# Stampa il report
print_analysis_report(
    REFERENCE_ATOM, reference_position, R_MIN, R_MAX, results, max_print=12
)

 ANALISI GEOMETRICA LOCALE

Atomo di riferimento : 200
Posizione [Å]        : (10.4010, 10.4010, 4.1604)
Intervallo           : 2.00 < d < 3.00 Å
Numero di vicini     : 12
Ambiente locale      : FCC bulk
d_nn                 : 2.9418 Å
R geometrico FCC     : 1.4709 Å
a FCC stimato        : 4.1604 Å

Primi vicini:
  Atom    84
  Atom   117
  Atom   118
  Atom   125
  Atom   187
  Atom   195
  Atom   199
  Atom   211
  Atom   222
  Atom   225
  Atom   277
  Atom   278


## 6. Visualizzazione con OVITO

Creiamo una pipeline OVITO con atomi colorati in base alla loro coordinazione:  
- **Giallo**: Atomo di riferimento  
- **Rosso**: Primi vicini (distanza tra 2.0 e 3.0 Å)  
- **Grigio chiaro**: Atomi di contesto (ridotti e semitrasparenti)

In [6]:
# Creiamo la pipeline OVITO
pipeline = create_ovito_pipeline(
    xyz_path,
    coords,
    neighbor_mask,
    reference_index,
    results["atomic_radius"],
    background_radius_scale=BACKGROUND_RADIUS_SCALE,
    background_transparency=BACKGROUND_TRANSPARENCY,
)

# Aggiungiamo alla scena
for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()

pipeline.add_to_scene()

print("✓ Pipeline OVITO creata e aggiunta alla scena.")

✓ Pipeline OVITO creata e aggiunta alla scena.


## 7. Pannello di visualizzazione

Mostriamo la struttura in 4 viste: Top (XY), Front (XZ), Left (YZ) e Perspective (3D).

In [7]:
grid = create_viewport_grid(R_MIN, R_MAX, REFERENCE_ATOM, show_context=True)
grid

GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

## NOTE: SI OSSERVI IL DODECAEDRO OSSERVATO NELLA VISTA PERSPECTIVE, CHE INDICA LA PRIMA SHELL DI COORDINAZIONE.

## 8. Analisi di atomi aggiuntivi

Ora analizziamo altri atomi per mostrare diverse situazioni (superficie, bordo, bulk).

### 8.1 Atomo 5 (superficie) - piccolo file

In [8]:
if FILENAME == "fcc100a108.txt":
    print("=" * 65)
    print(" ANALISI ATOMO 5 (SUPERFICIE)")
    print("=" * 65)

    results_5 = compute_neighbor_distances(coords, 5, R_MIN, R_MAX)
    reference_position_5 = coords[4]  # index 4 per atomo 5
    print_analysis_report(5, reference_position_5, R_MIN, R_MAX, results_5, max_print=8)

    print(f"\nInterpretazione:")
    print(
        f"• L'atomo 5 ha coordinazione {results_5['coordination_number']} (atteso 4 per singolo layer)."
    )
    print(
        f"• Si trova nella parte superiore della struttura (z = {reference_position_5[2]:.4f} Å)."
    )
else:
    print("Per vedere l'atomo 5, cambia FILENAME in 'fcc100a108.txt'.")

Per vedere l'atomo 5, cambia FILENAME in 'fcc100a108.txt'.


### 8.2 Atomo 75 (bulk) - piccolo file

In [9]:
if FILENAME == "fcc100a108.txt":
    print("=" * 65)
    print(" ANALISI ATOMO 75 (BULK)")
    print("=" * 65)

    results_75 = compute_neighbor_distances(coords, 75, R_MIN, R_MAX)
    reference_position_75 = coords[74]  # index 74 per atomo 75
    print_analysis_report(75, reference_position_75, R_MIN, R_MAX, results_75)

    print(f"\nInterpretazione:")
    print(
        f"• L'atomo 75 ha coordinazione {results_75['coordination_number']} (atteso 12 per FCC bulk)."
    )
    print(
        f"• Parametro reticolare stimato: {results_75['a_fcc']:.4f} Å (reale: 4.1604 Å)."
    )
else:
    print("Per vedere l'atomo 75, cambia FILENAME in 'fcc100a108.txt'.")

Per vedere l'atomo 75, cambia FILENAME in 'fcc100a108.txt'.


## 9. Riepilogo e commenti

### Cosa abbiamo imparato

1. **Metodo di analisi geometrica**: Calcolo di distanze, identificazione della prima shell, stima del parametro reticolare FCC.

2. **Classificazione dell'ambiente atomico**:
   - **Coordinazione 4**: singolo layer FCC(100)
   - **Coordinazione 8**: superficie FCC(100)
   - **Coordinazione 12**: FCC bulk

3. **Struttura cristallina FCC**:
   - Distanza primo vicino: $d_{\mathrm{nn}} = a / \sqrt{2}$
   - Raggio atomico geometrico: $R = d_{\mathrm{nn}} / 2$

### Limiti dell'analisi attuale

1. **Condizioni al contorno**: Non abbiamo applicato periodic boundary conditions (PBC). Nella minimum-image convention, alcune distanze apparirebbero diverse.

2. **Cutoff fisso**: Usiamo $2.0 < d < 3.0$ Å, che funziona per l'argento FCC ma dovrebbe essere adattato per altri materiali.

3. **Analisi locale**: Questo metodo non sostituisce tecniche strutturali complete come CNA (Common Neighbor Analysis) o PTM.

### Come estendere l'analisi

1. **Analizzare tutti gli atomi**: Calcolare la distribuzione dei numeri di coordinazione.
2. **Variare il cutoff**: Studiare come cambia la coordinazione al variare di $r_{\mathrm{cut}}$.
3. **Aggiungere PBC**: Implementare minimum-image convention per sistemi periodici.

---

*Notebook creato con il package `computational_course`*

In [10]:
# Verifica: le funzioni di utility sono accessibili
print("\nUtility disponibili:")
print("• find_project_root(): trova la root del progetto")
print("• load_structure(): carica coordinate da file")
print("• compute_neighbor_distances(): analisi geometrica")
print("• create_viewport_grid(): visualizzazione 2×2")
print("\n✓ Notebook completato.")


Utility disponibili:
• find_project_root(): trova la root del progetto
• load_structure(): carica coordinate da file
• compute_neighbor_distances(): analisi geometrica
• create_viewport_grid(): visualizzazione 2×2

✓ Notebook completato.


# NUOVA PARTE
we shall compute the distance between first, second, third neighbors in Ag fcc. ma si deve intrdurre un errore di 0.1 Å per simulare la precisione sperimentale.

POi si procederà con il pair potential analysis, calcolando l'energia di interazione tra gli atomi in base alla distanza e al potenziale scelto (ad esempio Lennard-Jones o EAM) e usare il concetto di cutoff.



In [11]:
n = 10
# first double loop example
mysum = 0
for i in range(1, n + 1):
    for j in range(1, n + 1):
        mysum = mysum + i + j
        print(f"i={i}, j={j}, mysum={mysum}")

# now we can use the same logic to compute the distances between first, second, and third neighbors in Ag fcc structure. We will introduce an error of 0.1 Å to simulate experimental precision.

error = 0.1  # Å
distances = []
for neighbor_index in results["neighbor_indices"]:
    neighbor_position = coords[neighbor_index]
    distance = np.linalg.norm(neighbor_position - reference_position)
    distance_with_error = distance + np.random.uniform(-error, error)
    distances.append(distance_with_error)
    print(
        f"Neighbor index: {neighbor_index + 1}, Distance: {distance:.4f} Å, With error: {distance_with_error:.4f} Å"
    )


# distanza tra i primi vicini, tra i secondi e i terzi vicini
first_neighbors = distances[:12]  # Assuming first shell has 12 neighbors
second_neighbors = distances[12:24]  # Assuming second shell has 12 neighbors
third_neighbors = distances[24:36]  # Assuming third shell has 12 neighbors


i=1, j=1, mysum=2
i=1, j=2, mysum=5
i=1, j=3, mysum=9
i=1, j=4, mysum=14
i=1, j=5, mysum=20
i=1, j=6, mysum=27
i=1, j=7, mysum=35
i=1, j=8, mysum=44
i=1, j=9, mysum=54
i=1, j=10, mysum=65
i=2, j=1, mysum=68
i=2, j=2, mysum=72
i=2, j=3, mysum=77
i=2, j=4, mysum=83
i=2, j=5, mysum=90
i=2, j=6, mysum=98
i=2, j=7, mysum=107
i=2, j=8, mysum=117
i=2, j=9, mysum=128
i=2, j=10, mysum=140
i=3, j=1, mysum=144
i=3, j=2, mysum=149
i=3, j=3, mysum=155
i=3, j=4, mysum=162
i=3, j=5, mysum=170
i=3, j=6, mysum=179
i=3, j=7, mysum=189
i=3, j=8, mysum=200
i=3, j=9, mysum=212
i=3, j=10, mysum=225
i=4, j=1, mysum=230
i=4, j=2, mysum=236
i=4, j=3, mysum=243
i=4, j=4, mysum=251
i=4, j=5, mysum=260
i=4, j=6, mysum=270
i=4, j=7, mysum=281
i=4, j=8, mysum=293
i=4, j=9, mysum=306
i=4, j=10, mysum=320
i=5, j=1, mysum=326
i=5, j=2, mysum=333
i=5, j=3, mysum=341
i=5, j=4, mysum=350
i=5, j=5, mysum=360
i=5, j=6, mysum=371
i=5, j=7, mysum=383
i=5, j=8, mysum=396
i=5, j=9, mysum=410
i=5, j=10, mysum=425
i=6, j=1, mysu

In [12]:
import numpy as np

filename = "/Users/francesco/Desktop/computational-course/data/raw/fcc100a108.txt"
pos = np.loadtxt(filename)
print("Total number of atoms:", pos.shape[0])
print("Y coordinate of atom 54:", pos[53, 1])
natoms = pos.shape[0]

# Parametro reticolare dell'Argento (Ag) in Å e distanze teoriche FCC
a = 4.08
d1_ref = a / (2**0.5)  # Primi vicini: ~2.885 Å
d2_ref = a  # Secondi vicini: ~4.080 Å
d3_ref = a * (1.5**0.5)  # Terzi vicini: ~4.997 Å
error = 0.1  # Tolleranza di 0.1 Å

all_distances = []
first_neighbors = []
second_neighbors = []
third_neighbors = []

# Nested loop su tutti gli atomi
for i in range(natoms):
    for j in range(natoms):
        if i != j:  # Esclude la distanza di un atomo con se stesso (d = 0)
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            distance = (dx**2 + dy**2 + dz**2) ** 0.5
            all_distances.append(distance)

            print(f"i={i}, j={j}, distance={distance:.4f} Å")

            # Filtro per primi, secondi e terzi vicini con tolleranza ±0.1 Å
            if abs(distance - d1_ref) <= error:
                first_neighbors.append(distance)
            elif abs(distance - d2_ref) <= error:
                second_neighbors.append(distance)
            elif abs(distance - d3_ref) <= error:
                third_neighbors.append(distance)

print(f"\nTotale coppie calcolate: {len(all_distances)}")
print(f"Coppie di primi vicini trovate: {len(first_neighbors)}")
print(f"Coppie di secondi vicini trovate: {len(second_neighbors)}")
print(f"Coppie di terzi vicini trovate: {len(third_neighbors)}")

Total number of atoms: 108
Y coordinate of atom 54: 10.401
i=0, j=1, distance=2.9418 Å
i=0, j=2, distance=2.9418 Å
i=0, j=3, distance=4.1604 Å
i=0, j=4, distance=6.5782 Å
i=0, j=5, distance=8.3208 Å
i=0, j=6, distance=2.9418 Å
i=0, j=7, distance=4.1604 Å
i=0, j=8, distance=2.9418 Å
i=0, j=9, distance=5.8837 Å
i=0, j=10, distance=6.5782 Å
i=0, j=11, distance=9.3029 Å
i=0, j=12, distance=6.5782 Å
i=0, j=13, distance=8.3208 Å
i=0, j=14, distance=6.5782 Å
i=0, j=15, distance=9.3029 Å
i=0, j=16, distance=8.8255 Å
i=0, j=17, distance=11.7674 Å
i=0, j=18, distance=5.0954 Å
i=0, j=19, distance=5.0954 Å
i=0, j=20, distance=6.5782 Å
i=0, j=21, distance=8.8255 Å
i=0, j=22, distance=7.7834 Å
i=0, j=23, distance=8.8255 Å
i=0, j=24, distance=2.9418 Å
i=0, j=25, distance=5.0954 Å
i=0, j=26, distance=8.8255 Å
i=0, j=27, distance=6.5782 Å
i=0, j=28, distance=5.0954 Å
i=0, j=29, distance=2.9418 Å
i=0, j=30, distance=8.8255 Å
i=0, j=31, distance=7.7834 Å
i=0, j=32, distance=2.9418 Å
i=0, j=33, distance=2

In [13]:
import numpy as np

# 1. Scegli l'atomo di riferimento (indice 0-based: es. 199 corrisponde all'Atomo 200)
# Se vuoi mantenere quello già selezionato nel notebook, lascia invariato reference_index
# reference_index = 199

natoms = coords.shape[0]
r_atom = results["atomic_radius"]

# Distanze teoriche FCC ricavate direttamente dal raggio atomico (a = 2*sqrt(2)*R)
d1_ref = 2.0 * r_atom  # Primi vicini (~2.9418 Å)
d2_ref = (8.0**0.5) * r_atom  # Secondi vicini (~4.1604 Å)
d3_ref = (12.0**0.5) * r_atom  # Terzi vicini (~5.0954 Å)
error = 0.1  # Tolleranza di 0.1 Å

# Inizializziamo le maschere booleane per le 3 shell
mask_1st = np.zeros(natoms, dtype=bool)
mask_2nd = np.zeros(natoms, dtype=bool)
mask_3rd = np.zeros(natoms, dtype=bool)

# 2. Ciclo su tutti gli atomi j rispetto all'atomo scelto (reference_index)
for j in range(natoms):
    if j != reference_index:
        dx = coords[reference_index, 0] - coords[j, 0]
        dy = coords[reference_index, 1] - coords[j, 1]
        dz = coords[reference_index, 2] - coords[j, 2]

        distance = (dx**2 + dy**2 + dz**2) ** 0.5

        if abs(distance - d1_ref) <= error:
            mask_1st[j] = True
        elif abs(distance - d2_ref) <= error:
            mask_2nd[j] = True
        elif abs(distance - d3_ref) <= error:
            mask_3rd[j] = True

print(f"Atomo scelto (indice {reference_index}, ID {reference_index + 1}):")
print(f"  • Primi vicini trovati   : {np.sum(mask_1st)} (d ≈ {d1_ref:.4f} Å)")
print(f"  • Secondi vicini trovati : {np.sum(mask_2nd)} (d ≈ {d2_ref:.4f} Å)")
print(f"  • Terzi vicini trovati   : {np.sum(mask_3rd)} (d ≈ {d3_ref:.4f} Å)")

# Uniamo le tre maschere (OR logico) per rendere visibili tutte e 3 le shell
combined_neighbor_mask = mask_1st | mask_2nd | mask_3rd

# 3. Creiamo la pipeline OVITO passando la maschera combinata
pipeline = create_ovito_pipeline(
    xyz_path,
    coords,
    combined_neighbor_mask,
    reference_index,
    results["atomic_radius"],
    background_radius_scale=BACKGROUND_RADIUS_SCALE,
    background_transparency=BACKGROUND_TRANSPARENCY,
)


# 4. Distinguiamo con colori diversi l'atomo centrale e le 3 shell di vicini
def color_neighbor_shells(frame, data):
    colors = data.particles_.create_property("Color")
    colors[reference_index] = (1.0, 0.15, 0.15)  # Rosso: Atomo di riferimento
    colors[mask_1st] = (0.20, 0.55, 1.0)  # Blu: Primi vicini (1ª shell)
    colors[mask_2nd] = (0.20, 0.80, 0.35)  # Verde: Secondi vicini (2ª shell)
    colors[mask_3rd] = (1.0, 0.80, 0.10)  # Giallo/Oro: Terzi vicini (3ª shell)


pipeline.modifiers.append(color_neighbor_shells)

# 5. Aggiungiamo alla scena rimuovendo le pipeline precedenti
for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()

pipeline.add_to_scene()

print("✓ Pipeline OVITO con 1°, 2° e 3° vicini creata e aggiunta alla scena.")

Atomo scelto (indice 199, ID 200):
  • Primi vicini trovati   : 12 (d ≈ 2.9418 Å)
  • Secondi vicini trovati : 6 (d ≈ 4.1604 Å)
  • Terzi vicini trovati   : 24 (d ≈ 5.0954 Å)
✓ Pipeline OVITO con 1°, 2° e 3° vicini creata e aggiunta alla scena.


In [14]:
grid = create_viewport_grid(R_MIN, R_MAX, REFERENCE_ATOM, show_context=True)
grid

GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

In [15]:
# versione del prof

import numpy as np

filename = "/Users/francesco/Desktop/computational-course/data/raw/fcc100a108.txt"
pos = np.loadtxt(filename)
natoms = pos.shape[0]
for i in range(0, natoms):
    for j in range(i + 1, natoms):
        dx = pos[i, 0] - pos[j, 0]
        dy = pos[i, 1] - pos[j, 1]
        dz = pos[i, 2] - pos[j, 2]
        dis = (dx**2 + dy**2 + dz**2) ** 0.5
        print(i, j, dis)


0 1 2.9418470524485123
0 2 2.9418470524485123
0 3 4.160399999999999
0 4 6.578169988682263
0 5 8.3208
0 6 2.9418470524485123
0 7 4.160399999999999
0 8 2.9418470524485123
0 9 5.883694104897024
0 10 6.578169988682263
0 11 9.302937213590125
0 12 6.578169988682263
0 13 8.3208
0 14 6.578169988682263
0 15 9.302937213590125
0 16 8.825541157345537
0 17 11.76738820979405
0 18 5.095428562937566
0 19 5.095428562937566
0 20 6.578169988682263
0 21 8.825541157345537
0 22 7.783395695967153
0 23 8.825541157345537
0 24 2.9418470524485123
0 25 5.095428562937566
0 26 8.825541157345537
0 27 6.578169988682263
0 28 5.095428562937566
0 29 2.9418470524485123
0 30 8.825541157345537
0 31 7.783395695967153
0 32 2.9418470524485123
0 33 2.9418470524485123
0 34 10.60698039217571
0 35 10.60698039217571
0 36 7.783395695967153
0 37 9.302937213590125
0 38 4.1604
0 39 5.095428562937567
0 40 7.783395695967153
0 41 9.302937213590125
0 42 5.095428562937567
0 43 7.206024179809556
0 44 5.095428562937567
0 45 5.883694104897024

## ORA TROVIAMO LA DISTANZA MINIMA tra tutte


In [16]:
import numpy as np

filename = "/Users/francesco/Desktop/computational-course/data/raw/fcc100a108.txt"
pos = np.loadtxt(filename)
natoms = pos.shape[0]
dmin = 10000
for i in range(0, natoms):
    for j in range(i + 1, natoms):
        if i != j:  # GUARDA LA LENNARD JONES SOMMA (i diverso da j)
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            dis = (dx**2 + dy**2 + dz**2) ** 0.5
            if dis < dmin:
                dmin = dis
                imin = i  # new distance estimate
                jmin = j
            print(
                i, j, dis
            )  # quindi troviamo la distanza minima tra tutti gli atomi, e gli indici degli atomi che la generano.

0 1 2.9418470524485123
0 2 2.9418470524485123
0 3 4.160399999999999
0 4 6.578169988682263
0 5 8.3208
0 6 2.9418470524485123
0 7 4.160399999999999
0 8 2.9418470524485123
0 9 5.883694104897024
0 10 6.578169988682263
0 11 9.302937213590125
0 12 6.578169988682263
0 13 8.3208
0 14 6.578169988682263
0 15 9.302937213590125
0 16 8.825541157345537
0 17 11.76738820979405
0 18 5.095428562937566
0 19 5.095428562937566
0 20 6.578169988682263
0 21 8.825541157345537
0 22 7.783395695967153
0 23 8.825541157345537
0 24 2.9418470524485123
0 25 5.095428562937566
0 26 8.825541157345537
0 27 6.578169988682263
0 28 5.095428562937566
0 29 2.9418470524485123
0 30 8.825541157345537
0 31 7.783395695967153
0 32 2.9418470524485123
0 33 2.9418470524485123
0 34 10.60698039217571
0 35 10.60698039217571
0 36 7.783395695967153
0 37 9.302937213590125
0 38 4.1604
0 39 5.095428562937567
0 40 7.783395695967153
0 41 9.302937213590125
0 42 5.095428562937567
0 43 7.206024179809556
0 44 5.095428562937567
0 45 5.883694104897024

# ora voglio computare la distanza TOTAL ENERGY (lennard-jones energy).

U = 1/2 sommatoria i != j a natoms di f(distance(i,j))
con f(distance) = 4*epsilon*((sigma/distance)**12 - (sigma/distance)**6)

CON esempio epsilon = 0.345 e sigma = 2.644
Per il momento non putiamo nessun radius di cutoff, ma si puo fare in seguito.




In [17]:
import numpy as np

# Parametri di Lennard-Jones per l'Argento (Ag) NO RADIUS CUTOFF
epsilon = 0.345  # eV
sigma = 2.644  # Å

natoms = pos.shape[0]
U_sum = 0.0

# Doppio ciclo su tutti gli atomi (senza raggio di cutoff)
for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            distance = (dx**2 + dy**2 + dz**2) ** 0.5

            # Potenziale di coppia f(distance) ASSUNTO LENNARD JONES 6-12
            sr6 = (sigma / distance) ** 6
            sr12 = sr6**2
            f_dist = 4.0 * epsilon * (sr12 - sr6)

            U_sum += f_dist

# Moltiplico per 1/2 per correggere il doppio conteggio (i,j) e (j,i)
U = 0.5 * U_sum

print(f"Numero totale di atomi (N)     : {natoms}")
print(f"Energia Totale LJ (U)          : {U:.4f} eV")
print(f"Energia media per atomo (U/N)  : {U / natoms:.4f} eV/atomo")

Numero totale di atomi (N)     : 108
Energia Totale LJ (U)          : -198.6561 eV
Energia media per atomo (U/N)  : -1.8394 eV/atomo


###   QUINDI: Energia Totale di Lennard-Jones

L'energia potenziale totale $U$ per un sistema di $N$ atomi (senza raggio di cutoff) è data da:

$$U = \frac{1}{2} \sum_{i=1}^{N} \sum_{\substack{j=1 \\ j \neq i}}^{N} f(r_{ij}), \qquad f(r_{ij}) = 4\epsilon \left[ \left(\frac{\sigma}{r_{ij}}\right)^{12} - \left(\frac{\sigma}{r_{ij}}\right)^{6} \right]$$

dove $r_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2 + (z_i - z_j)^2}$ è la distanza euclidea tra l'atomo $i$ e l'atomo $j$, e il fattore $\frac{1}{2}$ corregge il doppio conteggio delle coppie.

* **Parametri Argento (Ag FCC):** $\epsilon = 0.345\text{ eV}$, $\sigma = 2.644\text{ \AA}$
* **Risultati per il cluster ($N = 108$):**
  * Energia totale: $U = -198.6561\text{ eV}$
  * Energia media per atomo: $\frac{U}{N} = -1.8394\text{ eV/atomo}$

# PROVIAMO CON UN RAGGIO DI CUTOFF DI 3.0 Å


In [18]:
Rc = 3.0  # Å , U =....
# poi lo iteriamo per Rc=4 e Rc=6. dovremmo aspettarci che l'energia totale...

import numpy as np

# Parametri di Lennard-Jones per l'Argento (Ag)
epsilon = 0.345  # eV
sigma = 2.644  # Å
natoms = pos.shape[0]

# --- 1. Calcolo singolo per Rc = 3.0 Å ---
Rc = 3.0  # Å
U_sum = 0.0

for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            distance = (dx**2 + dy**2 + dz**2) ** 0.5

            # Applichiamo il raggio di cutoff Rc
            if distance <= Rc:
                sr6 = (sigma / distance) ** 6
                sr12 = sr6**2
                phi = 4.0 * epsilon * (sr12 - sr6)
                U_sum += phi

U = 0.5 * U_sum
print(f"Rc = {Rc:.1f} Å -> U = {U:.4f} eV | U/N = {U / natoms:.4f} eV/atomo\n")


# --- 2. Iterazione per Rc = 3.0, 4.0 e 6.0 Å ---
# Poi lo iteriamo per Rc=4.0 e Rc=6.0. Dovremmo aspettarci che l'energia totale:
# 1. Diventi più negativa (diminuisca) all'aumentare di Rc, perché per r > sigma (2.644 Å)
#    il potenziale di Lennard-Jones è sempre attrattivo (phi(r) < 0) e ogni nuova shell
#    di vicini inclusa aggiunge un contributo negativo che stabilizza il cristallo.
# 2. Converga rapidamente verso il valore senza cutoff (-198.6561 eV), poiché il potenziale
#    LJ è a corto raggio (~ 1/r^6) e i contributi delle shell lontane diventano via via trascurabili.
# 3. Vari "a gradini" solo quando Rc supera la distanza di una nuova shell del reticolo FCC.

cutoff_list = [
    2.0,
    2.9,
    3.0,
    4.0,
    5.0,
    6.0,
    7.0,
    8.0,
    12.0,
    14.0,
    16.0,
    20.0,
    30.0,
    50.0,
]

for Rc in cutoff_list:
    U_sum = 0.0
    pairs_counted = 0

    for i in range(natoms):
        for j in range(natoms):
            if i != j:
                dx = pos[i, 0] - pos[j, 0]
                dy = pos[i, 1] - pos[j, 1]
                dz = pos[i, 2] - pos[j, 2]
                distance = (dx**2 + dy**2 + dz**2) ** 0.5

                if distance <= Rc:
                    sr6 = (sigma / distance) ** 6
                    sr12 = sr6**2
                    U_sum += 4.0 * epsilon * (sr12 - sr6)
                    pairs_counted += 1

    U = 0.5 * U_sum
    print(
        f"Rc = {Rc:4.1f} Å | Coppie incluse: {pairs_counted:5d} | U = {U:9.4f} eV | U/N = {U / natoms:7.4f} eV/atomo"
    )

Rc = 3.0 Å -> U = -154.7958 eV | U/N = -1.4333 eV/atomo

Rc =  2.0 Å | Coppie incluse:     0 | U =    0.0000 eV | U/N =  0.0000 eV/atomo
Rc =  2.9 Å | Coppie incluse:     0 | U =    0.0000 eV | U/N =  0.0000 eV/atomo
Rc =  3.0 Å | Coppie incluse:   900 | U = -154.7958 eV | U/N = -1.4333 eV/atomo
Rc =  4.0 Å | Coppie incluse:   900 | U = -154.7958 eV | U/N = -1.4333 eV/atomo
Rc =  5.0 Å | Coppie incluse:  1332 | U = -173.1397 eV | U/N = -1.6031 eV/atomo
Rc =  6.0 Å | Coppie incluse:  3108 | U = -192.2330 eV | U/N = -1.7799 eV/atomo
Rc =  7.0 Å | Coppie incluse:  4188 | U = -195.3618 eV | U/N = -1.8089 eV/atomo
Rc =  8.0 Å | Coppie incluse:  5884 | U = -197.3161 eV | U/N = -1.8270 eV/atomo
Rc = 12.0 Å | Coppie incluse: 10120 | U = -198.5813 eV | U/N = -1.8387 eV/atomo
Rc = 14.0 Å | Coppie incluse: 11272 | U = -198.6500 eV | U/N = -1.8394 eV/atomo
Rc = 16.0 Å | Coppie incluse: 11532 | U = -198.6559 eV | U/N = -1.8394 eV/atomo
Rc = 20.0 Å | Coppie incluse: 11556 | U = -198.6561 eV | U/N = 

### Saturazione e Convergenza dell'Energia Totale al crescere di $R_c$

All'aumentare del raggio di cutoff $R_c$, l'energia totale di Lennard-Jones $U(R_c)$ si addensa fino a stabilizzarsi sul valore costante $U = -198.6561\text{ eV}$ per due ragioni:

1. **Decadimento fisico a corto raggio ($\propto r^{-6}$):**  
   Poiché l'integrale $\int_0^\infty 4\pi r^2 \phi(r)\,dr < \infty$ converge, il potenziale di Lennard-Jones è a corto raggio. Per $r_{ij} \gg \sigma$, il termine repulsivo diventa trascurabile e l'attrazione decade rapidamente come:
   $$\phi(r_{ij}) \approx -4\epsilon \left(\frac{\sigma}{r_{ij}}\right)^6$$
   Oltre $R_c \approx 10\text{–}12\text{ \AA}$, il contributo energetico di ciascuna nuova coppia è inferiore a $10^{-4}\text{ eV}$ (meno dello $0.03\%$ di un legame di primo vicino).

2. **Saturazione geometrica del cluster finito ($R_c \ge r_{\max}$):**  
   Il sistema analizzato (`fcc100a108.txt`) è un nanocristallo isolato di $N = 108$ atomi con lato $\frac{5}{2}a = 10.4010\text{ \AA}$. Esiste quindi una distanza massima finita $r_{\max}$ tra le coppie di atomi più lontane del reticolo:
   $$r_{\max} = \sqrt{\Delta x_{\max}^2 + \Delta y_{\max}^2 + \Delta z_{\max}^2} = 16.8996\text{ \AA}$$
   Per ogni $R_c \ge 16.8996\text{ \AA}$, la condizione $r_{ij} \le R_c$ include tutte le $N(N-1) = 11556$ coppie del sistema e l'energia rimane rigorosamente costante a $U = -198.6561\text{ eV}$.

In [19]:
# import numpy as np

epsilon = 0.345  # eV
sigma = 2.644  # Å
natoms = pos.shape[0]

# 1. Raccogliamo tutte le distanze e i contributi energetici di coppia (i != j)
distances = []
energies = []

for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            dist = (dx**2 + dy**2 + dz**2) ** 0.5

            sr6 = (sigma / dist) ** 6
            phi = 4.0 * epsilon * (sr6**2 - sr6)

            distances.append(dist)
            energies.append(0.5 * phi)

distances = np.array(distances)
energies = np.array(energies)

# 2. Ordiniamo le coppie per distanza crescente e calcoliamo l'energia cumulativa U(Rc)
sort_idx = np.argsort(distances)
dist_sorted = distances[sort_idx]
U_cumulative = np.cumsum(energies[sort_idx])
U_total = U_cumulative[-1]

# 3. Troviamo gli Rc critici di addensamento e di costanza assoluta
rc_99 = dist_sorted[np.where(np.abs(U_cumulative / U_total) >= 0.990)[0][0]]
rc_999 = dist_sorted[np.where(np.abs(U_cumulative / U_total) >= 0.999)[0][0]]
rc_4dec = dist_sorted[np.where(np.abs(U_cumulative - U_total) < 0.00005)[0][0]]
rc_max = dist_sorted[-1]

print(f"Energia Totale asintotica (U_tot)               : {U_total:.4f} eV")
print(f"• Rc per il 99.0% di U_tot (U = {0.990 * U_total:.4f} eV) : Rc = {rc_99:.4f} Å")
print(
    f"• Rc per il 99.9% di U_tot (U = {0.999 * U_total:.4f} eV) : Rc = {rc_999:.4f} Å"
)
print(
    f"• Rc da cui U è fisso a -198.6561 eV (4 dec.)   : Rc = {rc_4dec:.4f} Å (esatto: r_max = {rc_max:.4f} Å)"
)

Energia Totale asintotica (U_tot)               : -198.6561 eV
• Rc per il 99.0% di U_tot (U = -196.6696 eV) : Rc = 7.7834 Å
• Rc per il 99.9% di U_tot (U = -198.4575 eV) : Rc = 10.6070 Å
• Rc da cui U è fisso a -198.6561 eV (4 dec.)   : Rc = 16.8996 Å (esatto: r_max = 16.8996 Å)


### noi usaremo Rc tra 2 e 3 angstroms (di altri dati) ecc.

------------------------------------------------

In [20]:
# quanti vicini?

import numpy as np

R = 4.6  # Raggio di vicinato in Å

natoms = pos.shape[0]
nnbrs = np.zeros(natoms, dtype=int)

for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            distance = np.sqrt(dx**2 + dy**2 + dz**2)

            if distance < R:
                nnbrs[i] += 1

print("Numero di atomi:", natoms)
print("Numero di vicini per ciascun atomo:", nnbrs)

Numero di atomi: 108
Numero di vicini per ciascun atomo: [11  6  9 12  8  8  9 12 13 13 12  9  8  8 12  9 11  6 17 17 15  8 16 12
 16 12  8 15 12 16 12 16 11 11 11 11 12 12 16  9 12 12 18 18 13 17 17 13
 13 17 17 13 16  9  9 16 17 13 12 12 17 13 13 17 16  9 18 18 13 17 12 12
 16 12 17 17 12 16 16 12 11 11 15  8 11 11 12 16 15  8  9 12 11  6  9 12
 13 13  6 11 12  9 12  9  8  8  8  8]


In [21]:
# quale vicino?

R = 4.6
atom_i = 1

vicini = []

for j in range(natoms):
    if j != atom_i:
        dx = pos[atom_i, 0] - pos[j, 0]
        dy = pos[atom_i, 1] - pos[j, 1]
        dz = pos[atom_i, 2] - pos[j, 2]

        distance = np.sqrt(dx**2 + dy**2 + dz**2)

        if distance < R:
            vicini.append(j)

print(f"Atomo selezionato: {atom_i}")
print(f"Numero di vicini: {len(vicini)}")
print(f"Indici dei vicini: {vicini}")

Atomo selezionato: 1
Numero di vicini: 6
Indici dei vicini: [0, 2, 6, 32, 33, 39]


In [22]:
# wich NBRG of atom[12(1),0] is = 4?
# --- atom[12(2),0] is =

# wich NBRS of atom[i, NNBRS[i]-1] = j?


# Inizializzazione
force = np.zeros((natoms, 3))
epot = 0.0
nnbrs = np.zeros(natoms, dtype=int)
whichnbr = np.zeros((natoms, natoms), dtype=int)  # questo è DA CAPIRE.

rc = 3

# 1. Calcolo dei vicini entro il cutoff rc
for i in range(natoms):
    for j in range(natoms):
        if i != j:
            rij = pos[i] - pos[j]
            dist = np.linalg.norm(rij)

            if dist < rc:
                whichnbr[i, nnbrs[i]] = j  # QUESTO è DA CAPIRE
                nnbrs[i] += 1


# 2. Calcolo dell'energia potenziale considerando, per OGNI ATOMO,
# esclusivamente le interazioni con i suoi vicini già individuati
for i in range(natoms):
    for k in range(nnbrs[i]):
        j = whichnbr[i, k]  # DA CAPIRE

        rij = pos[i] - pos[j]
        dist = np.linalg.norm(rij)

        sr6 = (sigma / dist) ** 6
        phi = 4.0 * epsilon * (sr6**2 - sr6)

        # Dividiamo per 2 perché ogni coppia è contata due volte
        epot += 0.5 * phi


# Risultati

print(nnbrs)
print("\n")
print(whichnbr)
print("\n")
print(f"Numero di atomi                 : {natoms}")
print(f"Energia potenziale totale       : {epot:.6f} eV")
print(f"Numero medio di vicini          : {np.mean(nnbrs):.2f}")

[ 8  3  5  8  5  5  5  8  8  8  8  5  5  5  8  5  8  3 12 12 12  5 12  8
 12  8  5 12  8 12  8 12  8  8  8  8  8  8 12  5  8  8 12 12  8 12 12  8
  8 12 12  8 12  5  5 12 12  8  8  8 12  8  8 12 12  5 12 12  8 12  8  8
 12  8 12 12  8 12 12  8  8  8 12  5  8  8  8 12 12  5  5  8  8  3  5  8
  8  8  3  8  8  5  8  5  5  5  5  5]


[[ 1  2  6 ...  0  0  0]
 [ 0 32 33 ...  0  0  0]
 [ 0  3 24 ...  0  0  0]
 ...
 [76 82 83 ...  0  0  0]
 [79 88 89 ...  0  0  0]
 [73 88 89 ...  0  0  0]]


Numero di atomi                 : 108
Energia potenziale totale       : -154.795762 eV
Numero medio di vicini          : 8.33


# QUINDI IMPORTANTE:

nnbrs è un array che registra quanti vicini ha ciascun atomo, mentre whichnbr è una matrice che memorizza gli indici degli atomi vicini. Ogni riga di whichnbr corrisponde a un atomo e ogni elemento della riga contiene l'indice di un suo vicino. Inizialmente entrambe le strutture sono impostate a zero; quando viene trovato un atomo a distanza inferiore al cutoff, il suo indice viene salvato in whichnbr e il contatore nnbrs viene incrementato.

Gli zeri rimanenti in ogni riga sono semplicemente valori iniziali non utilizzati: per sapere quali elementi sono validi, si guarda nnbrs[i]



-----


Questo approccio serve a ridurre il numero di interazioni da calcolare e rendere più efficiente la simulazione atomistica.
Primo ciclo: confronti ogni atomo con tutti gli altri per identificare quelli che si trovano entro il cutoff rc.

Secondo ciclo: calcoli l'energia potenziale considerando soltanto le coppie di atomi vicini già individuate, senza ripetere il confronto con tutti gli atomi del sistema.

Vantaggio: per sistemi con molti atomi, questo metodo può ridurre notevolmente il costo computazionale, soprattutto se ogni atomo ha pochi vicini rispetto al numero totale di atomi.

In sostanza, separi la ricerca dei vicini dal calcolo delle interazioni. Questo è utile perché il potenziale di Lennard-Jones dipende dalla distanza tra le coppie di atomi e, con un cutoff, puoi trascurare le interazioni oltre la distanza scelta.
Nota: nel tuo codice attuale, la ricerca iniziale confronta ancora tutte le coppie di atomi; il vantaggio si ottiene soprattutto quando riutilizzi la lista dei vicini per calcolare energia e forze, senza ricostruirla ogni volta.


----


Nel mio codice, con rc = 3.0 Å:
Se la distanza tra due atomi è minore di 3.0 Å, vengono considerati vicini e la loro interazione contribuisce all'energia potenziale.
Se la distanza è maggiore o uguale a 3.0 Å, quella coppia NON viene inserita nella lista dei vicini e la sua INTERAZIONE VIENE IGNORATA.
Esempio: l'atomo 0 può avere due vicini entro il cutoff e altri tre atomi più lontani. L'atomo 0 rimane nel sistema, ma soltanto le interazioni con i due vicini vengono considerate nel calcolo.

In [23]:
# dinamical double loop:

# Code which computes nbrs, then computes forces based on nbrs, then epot based on nbrs (1/3)

natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)
rc = 3.0
eps = 0.345
sigma = 2.644
# 1) I compute the nbrs
for i in range(0, natoms):
    for j in range(0, natoms):
        # K = wichnbr[i,j]
        dx = pos[i, 0] - pos[j, 0]
        dy = pos[i, 1] - pos[j, 1]
        dz = pos[i, 2] - pos[j, 2]
        d2 = dx * dx + dy * dy + dz * dz
        dis = d2**0.5
        if dis < (rc) and i != j:
            nnbrs[i] = nnbrs[i] + 1
            whichnbr[i, nnbrs[i] - 1] = j
# DISTANCE BETWEEN ATOM AND THE NBRS OF ANTOHER ATHOM

In [24]:
# dinamical double loop:

natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)
rc = 3.0
eps = 0.345
sigma = 2.644

for i in range(0, natoms):
    for j in range(0, natoms):
        k = wichnbr[i, j]
        dx = pos[i, 0] - pos[k, 0]
        dy = pos[i, 1] - pos[k, 1]
        dz = pos[i, 2] - pos[k, 2]
        d2 = dx * dx + dy * dy + dz * dz
        dis = d2**0.5
        epot = ...  # lennard jones

# DISTANCE BETWEEN ATOM AND THE NBRS OF ANTOHER ATHOM

NameError: name 'wichnbr' is not defined

In [ ]:
# codice corretto:


natoms = pos.shape[0]
force = np.zeros((natoms, 3))
nnbrs = np.zeros(natoms, int)
whichnbr = np.zeros((natoms, natoms), int)

rc = 4.0
eps = 0.345
sigma = 2.644

# 1. Calcolo dei vicini
for i in range(natoms):
    for j in range(natoms):
        if i != j:
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]

            d2 = dx**2 + dy**2 + dz**2
            dis = d2**0.5

            if dis < rc:
                whichnbr[i, nnbrs[i]] = j
                nnbrs[i] += 1

# 2. Calcolo dell'energia potenziale usando i vicini
epot = 0.0

for i in range(natoms):
    for j in range(nnbrs[i]):
        k = whichnbr[i, j]

        dx = pos[i, 0] - pos[k, 0]
        dy = pos[i, 1] - pos[k, 1]
        dz = pos[i, 2] - pos[k, 2]

        d2 = dx**2 + dy**2 + dz**2
        dis = d2**0.5

        sr6 = (sigma / dis) ** 6
        phi = 4 * eps * (sr6**2 - sr6)

        epot += 0.5 * phi

print("Energia potenziale totale:", epot, "eV")
print("Numero medio di vicini:", np.mean(nnbrs))


Energia potenziale totale: -154.7957616328464 eV
Numero medio di vicini: 8.333333333333334


# Legenda: lista dei vicini e energia di Lennard-Jones

## 1. Variabili

| Nome | Tipo / forma | Significato |
|---|---|---|
| `natoms` | int (108) | numero di atomi |
| `pos` | array `(natoms, 3)` | coordinate x, y, z di ogni atomo |
| `sigma`, `epsilon` | float | parametri di Lennard-Jones (distanza e profondità della buca) |
| `rc` | float (3.0) | raggio di cutoff: oltre questa distanza due atomi non sono vicini |
| `nnbrs` | array `(natoms,)` | `nnbrs[i]` = **quanti** vicini ha l'atomo `i` |
| `whichnbr` | array `(natoms, natoms)` | `whichnbr[i, k]` = **indice** del k-esimo vicino dell'atomo `i` |
| `epot` | float | energia potenziale totale accumulata |
| `force` | array `(natoms, 3)` | inizializzata ma **non usata** in questo frammento |

## 2. Idea chiave: `nnbrs[i]` fa due lavori

```python
whichnbr[i, nnbrs[i]] = j   # scrive j nel prossimo slot libero della riga i
nnbrs[i] += 1               # sposta il contatore allo slot successivo
```

`nnbrs[i]` è insieme il **contatore** dei vicini trovati e l'**indice del prossimo slot libero**.
A fine ciclo coincide con il numero totale di vicini.

## 3. Come leggere `whichnbr`

- Una **riga per atomo**, una **colonna per vicino**.
- Sono validi **solo** i primi `nnbrs[i]` elementi di ogni riga:

```python
whichnbr[i, :nnbrs[i]]      # vicini veri dell'atomo i
whichnbr[i, nnbrs[i]-1]     # ultimo vicino valido
```

- Gli `0 0 0` a destra sono **riempimento** (da `np.zeros`), non "vicino = atomo 0".
- Esempio: riga `[0 32 33 ... 0 0 0]` con `nnbrs = 3` significa che i vicini sono 0, 32, 33.

## 4. Passi dell'algoritmo

1. **Costruzione della lista:** per ogni coppia `(i, j)` con `i != j` calcola
   $r_{ij} = |\vec r_i - \vec r_j|$; se `dist < rc`, aggiungi `j` ai vicini di `i`.
2. **Energia:** per ogni atomo `i` e per ogni suo vicino `j = whichnbr[i, k]`, `k = 0 … nnbrs[i]-1`:

$$\varphi(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

   Nel codice `sr6 = (sigma/dist)**6`, quindi `sr6**2` è il termine a potenza 12.
3. **Fattore 0.5:** ogni coppia $(i,j)$ compare due volte (in riga `i` e in riga `j`), quindi si somma $\tfrac12 \varphi$.

$$E_{pot} = \sum_i \sum_{j \in \text{vicini}(i)} \tfrac12\,\varphi(r_{ij})$$

## 5. Come leggere l'output

| Stampa | Cosa è |
|---|---|
| matrice grande | `whichnbr` (108 × 108, abbreviata con `...`) |
| vettore di interi | `nnbrs`: numero di vicini di ciascun atomo |
| `Numero di atomi` | `natoms` |
| `Energia potenziale totale` | `epot` in eV |
| `Numero medio di vicini` | `np.mean(nnbrs)`, la coordinazione media |

## 6. Controlli di coerenza

- **Simmetria:** se `j` è nella riga di `i`, allora `i` è nella riga di `j`.
- **Doppio conteggio:** `nnbrs.sum() / 2` = numero di coppie distinte (con 8.33 di media su 108 atomi, circa 450).
- **Coordinazione:** valori alti (es. 12) indicano atomi interni, valori bassi (3, 5, 8) atomi di superficie, spigolo o vertice.

## 7. Limiti del codice

- Costo $O(N^2)$: scorre tutte le coppie. Va bene per 108 atomi, non per sistemi grandi.
- Nessuna **condizione periodica al contorno**: le distanze sono quelle dirette, senza convenzione dell'immagine minima.
- Il cutoff è **troncato** (nessun shift dell'energia a `rc`), quindi `epot` dipende un po' dalla scelta di `rc`.

### NELLA DIMANICA, I VICINI CAMBIANO.


In [ ]:
import numpy as np
import pandas as pd

natoms = pos.shape[0]
rc = 3.0  # sigma ed epsilon restano quelli già definiti

# 1. Distanze tra tutte le coppie (matrice natoms x natoms)
diff = pos[:, None, :] - pos[None, :, :]
dist = np.linalg.norm(diff, axis=2)

# 2. Tabella delle coppie entro il cutoff, escludendo i == j
i_idx, j_idx = np.where((dist < rc) & ~np.eye(natoms, dtype=bool))
pairs = pd.DataFrame({"i": i_idx, "j": j_idx, "dist": dist[i_idx, j_idx]})

# 3. Energia di Lennard-Jones per ogni coppia
sr6 = (sigma / pairs["dist"]) ** 6
pairs["phi"] = 4.0 * epsilon * (sr6**2 - sr6)

# Ogni coppia compare due volte (i->j e j->i), quindi 0.5
epot = 0.5 * pairs["phi"].sum()

# 4. nnbrs: numero di vicini per atomo (0 se l'atomo è isolato)
nnbrs = pairs.groupby("i").size().reindex(range(natoms), fill_value=0)

# 5. whichnbr: una riga per atomo, colonna k = k-esimo vicino
pairs["k"] = pairs.groupby("i").cumcount()
whichnbr = (
    pairs.pivot(index="i", columns="k", values="j")
    .reindex(range(natoms))
    .astype("Int64")
)  # <NA> dove non c'è vicino
whichnbr.insert(0, "nnbrs", nnbrs)
whichnbr.index.name = "atomo"

# Risultati
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(whichnbr)

print(f"Numero di atomi            : {natoms}")
print(f"Energia potenziale totale  : {epot:.6f} eV")
print(f"Numero medio di vicini     : {nnbrs.mean():.2f}")


# quelli che hanno pochi VICINI sono quelli di solito ai lati del cubo e se hanno pochi vicini danno un contributo alla interazione minore rispetto agli altri . in dinamica vedremo che quelli che hanno pochi vicini si muoveranno tanto e a volte si staccheranno dal cubo. quelli invece nel bulk (con 12 primi vicini ad esempio) si muovono poco.

k,nnbrs,0,1,2,3,4,5,6,7,8,9,10,11
atomo,,,,,,,,,,,,,
0,8,1,2,6,8,24,29,32,33,<NA>,<NA>,<NA>,<NA>
1,3,0,32,33,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,5,0,3,24,25,32,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,8,2,4,8,10,19,20,24,25,<NA>,<NA>,<NA>,<NA>
4,5,3,5,20,21,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,5,4,10,20,21,23,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,5,0,7,28,29,33,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,8,6,8,12,14,18,27,28,29,<NA>,<NA>,<NA>,<NA>
8,8,0,3,7,9,18,19,24,29,<NA>,<NA>,<NA>,<NA>


Numero di atomi            : 108
Energia potenziale totale  : -154.795762 eV
Numero medio di vicini     : 8.33
